# Grafici sulla demografia

Questa categoria genera grafici su popolazione, fasce di eta, natalita, dipendenze, migrazioni e aspettativa di vita. Dove WDI non basta, la specifica segnala UN WPP o ISTAT.

Ogni grafico riporta la fonte primaria usata nella generazione e la dicitura `elaborazione Nazareno Lecis`.


## Come vengono generati

Il notebook separa tre cose: la specifica del grafico, la fonte dati e la trasformazione. Ogni specifica dichiara `titolo`, `tipo_grafico`, `anno_inizio`, `ultimo_dato`, fonte primaria, fonti alternative e cosa viene mostrato. `definisci_grafico_da_indicatore_world_bank` usa direttamente un indicatore WDI; `definisci_grafico_da_rapporto_world_bank` combina due serie WDI; `registra_grafico_da_collegare_a_api` mantiene in inventario un grafico per cui la fonte pubblica esiste ma non e ancora mappata nel codice.

Quando il grafico viene generato, `genera_grafici_e_inventario` scarica i dati via API, applica la trasformazione indicata, costruisce il confronto con Italia, min-max, percentili 25-75 e mediana dei paesi avanzati quando disponibile, poi mostra l'inventario e lascia il plot nel notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analisi.utils.grafici import (
    PAESI_MONDO,
    definisci_grafico_da_rapporto_world_bank,
    definisci_grafico_da_indicatore_world_bank,
    genera_grafici_e_inventario,
    registra_grafico_da_collegare_a_api,
)

from sources.api_catalog import FONTI_API


## Fonti disponibili per questa categoria

La tabella sotto elenca le API ufficiali considerate per la categoria. Una stessa variabile puo comparire in piu fonti: il notebook indica una fonte primaria per la generazione, ma mantiene le alternative quando esistono definizioni ufficiali equivalenti o vicine.


In [ ]:
fonti_categoria = ('ISTAT SDMX API', 'UN WPP Data Portal API', 'World Bank API', 'OECD SDMX API')
catalogo_fonti_api = pd.DataFrame(
    [
        {
            "fonte": fonte.nome,
            "ente": fonte.ente,
            "aree": ", ".join(fonte.aree),
            "note": fonte.note,
            "endpoint_controllo": fonte.url_controllo,
        }
        for fonte in FONTI_API
        if fonte.nome in fonti_categoria
    ]
)
catalogo_fonti_api


## Specifiche dei grafici

Le righe sotto sono volutamente esplicite: per ogni grafico si legge il titolo dell'analisi, il tipo di grafico, la fonte primaria, le fonti ufficiali alternative, l'anno di partenza, l'ultimo dato richiesto e la trasformazione applicata.


In [ ]:
SPECIFICHE_GRAFICI = [
    definisci_grafico_da_indicatore_world_bank(
        titolo='Popolazione totale - livello',
        nome_output='popolazione_totale_livello',
        indicatore='SP.POP.TOTL',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione totale - livello: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Popolazione totale - crescita media',
        nome_output='popolazione_totale_crescita_media',
        indicatore='SP.POP.TOTL',
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione totale - crescita media: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Popolazione totale - crescita cumulata',
        nome_output='popolazione_totale_crescita_cumulata',
        indicatore='SP.POP.TOTL',
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione totale - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Popolazione 15-64 anni - livello',
        nome_output='popolazione_15_64_livello',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.1564.TO.ZS',
        scala=0.01,
        trasformazione='ratio',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione 15-64 anni - livello: serie calcolata moltiplicando SP.POP.TOTL per SP.POP.1564.TO.ZS e scala 0.01, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Popolazione 15-64 anni - crescita media',
        nome_output='popolazione_15_64_crescita_media',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.1564.TO.ZS',
        scala=0.01,
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione 15-64 anni - crescita media: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Popolazione 15-64 anni - crescita cumulata',
        nome_output='popolazione_15_64_crescita_cumulata',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.1564.TO.ZS',
        scala=0.01,
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione 15-64 anni - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Popolazione 15-74 anni',
        nome_output='popolazione_15_74',
        fonte_primaria='UN WPP Data Portal API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Popolazione 15-74 anni: Richiede serie per fasce 15-74 via API demografica.',
        fonti_alternative=('ISTAT SDMX API', 'OECD SDMX API'),
        note='Richiede serie per fasce 15-74 via API demografica.',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Popolazione prime age 25-49',
        nome_output='popolazione_prime_age_25_49',
        fonte_primaria='UN WPP Data Portal API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Popolazione prime age 25-49: Richiede serie per fasce 25-49 via API demografica.',
        fonti_alternative=('ISTAT SDMX API', 'OECD SDMX API'),
        note='Richiede serie per fasce 25-49 via API demografica.',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Bambini 0-14 anni - livello',
        nome_output='bambini_0_14_livello',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.0014.TO.ZS',
        scala=0.01,
        trasformazione='ratio',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Bambini 0-14 anni - livello: serie calcolata moltiplicando SP.POP.TOTL per SP.POP.0014.TO.ZS e scala 0.01, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Bambini 0-14 anni - crescita media',
        nome_output='bambini_0_14_crescita_media',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.0014.TO.ZS',
        scala=0.01,
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Bambini 0-14 anni - crescita media: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Bambini 0-14 anni - crescita cumulata',
        nome_output='bambini_0_14_crescita_cumulata',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.0014.TO.ZS',
        scala=0.01,
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Bambini 0-14 anni - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Popolazione anziana 65+ - livello',
        nome_output='popolazione_anziana_livello',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.65UP.TO.ZS',
        scala=0.01,
        trasformazione='ratio',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione anziana 65+ - livello: serie calcolata moltiplicando SP.POP.TOTL per SP.POP.65UP.TO.ZS e scala 0.01, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Popolazione anziana 65+ - crescita media',
        nome_output='popolazione_anziana_crescita_media',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.65UP.TO.ZS',
        scala=0.01,
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione anziana 65+ - crescita media: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    definisci_grafico_da_rapporto_world_bank(
        titolo='Popolazione anziana 65+ - crescita cumulata',
        nome_output='popolazione_anziana_crescita_cumulata',
        numeratore='SP.POP.TOTL',
        denominatore='SP.POP.65UP.TO.ZS',
        scala=0.01,
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione anziana 65+ - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        operazione='product',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Popolazione femminile in eta fertile',
        nome_output='popolazione_femminile_eta_fertile',
        fonte_primaria='UN WPP Data Portal API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Popolazione femminile in eta fertile: Richiede serie femminili per eta fertile.',
        fonti_alternative=('ISTAT SDMX API', 'OECD SDMX API'),
        note='Richiede serie femminili per eta fertile.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Tasso di natalita',
        nome_output='tasso_natalita',
        indicatore='SP.DYN.CBRT.IN',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Tasso di natalita: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Fecondita - nati vivi per donna',
        nome_output='fecondita_nati_per_donna',
        indicatore='SP.DYN.TFRT.IN',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Fecondita - nati vivi per donna: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Rapporto dipendenza anziani',
        nome_output='rapporto_dipendenza_anziani',
        indicatore='SP.POP.DPND.OL',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Rapporto dipendenza anziani: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Rapporto dipendenza bambini',
        nome_output='rapporto_dipendenza_bambini',
        indicatore='SP.POP.DPND.YG',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Rapporto dipendenza bambini: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Popolazione 0-14 anni - % sul totale',
        nome_output='popolazione_0_14_quota',
        indicatore='SP.POP.0014.TO.ZS',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione 0-14 anni - % sul totale: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Popolazione 65+ anni - % sul totale',
        nome_output='popolazione_65_plus_quota',
        indicatore='SP.POP.65UP.TO.ZS',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Popolazione 65+ anni - % sul totale: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Struttura popolazione per eta dettagliata',
        nome_output='struttura_popolazione_eta',
        fonte_primaria='UN WPP Data Portal API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Struttura popolazione per eta dettagliata: Richiede fasce 0-4, 0-19, 25-44, 50-64.',
        fonti_alternative=('ISTAT SDMX API', 'OECD SDMX API'),
        note='Richiede fasce 0-4, 0-19, 25-44, 50-64.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Saldo naturale proxy: natalita meno mortalita',
        nome_output='saldo_naturale_proxy',
        indicatore='SP.DYN.CBRT.IN',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Saldo naturale proxy: natalita meno mortalita: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
        note='Nel notebook il saldo naturale va letto insieme a SP.DYN.CDRT.IN.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Saldo migratorio totale',
        nome_output='saldo_migratorio_totale',
        indicatore='SM.POP.NETM',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Saldo migratorio totale: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Saldo migratorio italiani ed espatri',
        nome_output='saldo_migratorio_italiani',
        fonte_primaria='ISTAT SDMX API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Saldo migratorio italiani ed espatri: Richiede dataset ISTAT migrazioni cittadini italiani.',
        fonti_alternative=('UN WPP Data Portal API', 'OECD SDMX API'),
        note='Richiede dataset ISTAT migrazioni cittadini italiani.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Aspettativa di vita',
        nome_output='aspettativa_vita',
        indicatore='SP.DYN.LE00.IN',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Aspettativa di vita: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('UN WPP Data Portal API', 'ISTAT SDMX API', 'OECD SDMX API'),
    ),
]

print(f"Specifiche nella categoria: {len(SPECIFICHE_GRAFICI)}")


## Inventario e generazione

La tabella risultante mostra cosa viene prodotto, quale fonte e stata usata, quali alternative sono possibili, da quale anno parte la serie, fino a quale ultimo dato viene richiesto e quali grafici restano da collegare.


In [ ]:
cartella_output = ROOT / "analisi" / "demografia"
percorsi, inventario = genera_grafici_e_inventario(SPECIFICHE_GRAFICI, cartella_output)

colonne = [
    "titolo",
    "tipo_grafico",
    "cosa_mostra",
    "fonte_primaria",
    "fonti_alternative",
    "anno_inizio",
    "ultimo_dato",
    "trasformazione",
    "confronto",
    "stato",
    "nome_output",
    "note",
    "errore",
]
print(f"PNG generati: {len(percorsi)}")
for percorso in percorsi:
    print(percorso)
inventario[colonne]


## Plot dei grafici generati

I plot visualizzati qui sono quelli creati dalla cella precedente. I PNG locali restano fuori da Git.


In [ ]:
from IPython.display import Image, display

if not percorsi:
    print("Nessun PNG generato: tutti i grafici della categoria sono ancora da collegare o la fonte non ha restituito dati.")
for percorso in percorsi:
    display(Image(filename=str(percorso)))
